# 은닉 뉴런 수 / 정답 품질 비교

같은 자가대전 데이터로 신경망을 학습하고, 선형 평가와 대국시켜 점수를 비교합니다.
런타임은 **CPU** 로 두세요 (GPU 는 쓰지 않습니다). 위에서부터 한 셀씩 실행하고, 끊기면 셀 1·2 를 다시 실행한 뒤 끊긴 셀부터 이어 가세요.

- 정답 품질 실험: 셀 1 의 `TIME_MS` 를 30 (기준) 과 300 (깊은 탐색) 으로 각각 한 번씩 돌리고 점수를 비교합니다. 빌드와 코드 받기(셀 3·4)는 한 번만 하면 됩니다.
- 크기 비교: `HIDDEN_LIST = [32, 256]`.
- 마지막 셀이 결과를 Google Drive 에 저장합니다.

In [ ]:
import os, json, shutil, subprocess, pathlib, time

REPO_URL = "https://github.com/Perdume/Slider.git"
BRANCH = "rust-port"
HIDDEN_LIST = [32]        # 비교할 은닉 뉴런 수 (크기를 비교하려면 [32, 256])
GAMES = 200               # 자가대전 판 수 (데이터 양)
TIME_MS = 300             # 자가대전 수당 시간(ms). 정답(탐색 점수)의 품질을 정한다. 기준 실험은 30
EPOCHS = 40
MATCH_GAMES = 100         # 크기별 대국 수 (짝수)
MATCH_DEPTH = 5           # 숫자면 깊이 고정 대국, None 이면 시간 제한 대국
MATCH_TIME_MS = 50        # MATCH_DEPTH 가 None 일 때만 사용
SEED = 1
WEIGHTS_PATH = None       # 튜닝한 weights.json 을 올렸으면 경로, 없으면 None (기본 가중치)

THREADS = os.cpu_count() or 2
WORK = pathlib.Path("/content/hidden_cmp"); WORK.mkdir(parents=True, exist_ok=True)
W = WEIGHTS_PATH if WEIGHTS_PATH and os.path.exists(WEIGHTS_PATH) else "default"
os.environ["PATH"] = str(pathlib.Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
print("threads:", THREADS, "| 선형 가중치:", W)

In [ ]:
def sh(cmd, cwd=None, env=None):
    print("$", " ".join(map(str, cmd)) if isinstance(cmd, list) else cmd, flush=True)
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True, cwd=cwd, env=env)


def run(binary, args):
    cmd = [str(binary), *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        line = line.rstrip("\n"); lines.append(line); print("   ", line, flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"실행 실패 (exit {p.returncode}): {cmd[0]}")
    return lines

## 3. Rust 설치와 코드 받기

In [ ]:
if shutil.which("cargo") is None:
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
REPO = WORK / "Slider"
if not REPO.exists():
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO])
print("코드 준비됨:", REPO)

## 4. 은닉 크기별 빌드 (크기는 빌드할 때 정해집니다)

In [ ]:
BIN = {}
for h in HIDDEN_LIST:
    target = WORK / f"target_{h}"
    sh(["cargo", "build", "--release", "--bin", "selfplay", "--bin", "trainnet", "--bin", "matchup"],
       cwd=REPO / "rust", env={**os.environ, "SLIDER_HIDDEN": str(h), "CARGO_TARGET_DIR": str(target)})
    BIN[h] = target / "release"
print("빌드 완료:", {h: str(p) for h, p in BIN.items()})

## 5. 자가대전 데이터 (이미 있으면 건너뜀. 신경망 없이 두므로 은닉 크기와 무관)

In [ ]:
data = WORK / f"selfplay_{GAMES}g_{TIME_MS}ms.jsonl"
if data.exists() and data.stat().st_size > 0:
    print("이미 있음:", data, f"({data.stat().st_size / 1e6:.1f}MB)")
else:
    t0 = time.time()
    run(BIN[HIDDEN_LIST[0]] / "selfplay", ["--games", GAMES, "--time-ms", TIME_MS, "--random-plies", 4, "--epsilon", 0.05,
                                           "--threads", THREADS, "--seed", SEED, "--weights", W, "--positions", "yes", "--out", data])
    print(f"자가대전 {time.time() - t0:.0f}초")

## 6. 학습 + 대국 함수

In [ ]:
results = {}


def run_one(h):
    print(f"\n=============== 은닉 {h} ===============")
    net = WORK / f"net_{h}.txt"
    train_lines = run(BIN[h] / "trainnet", ["--data", data, "--weights", W, "--out", net, "--epochs", EPOCHS, "--lambda", 1.0, "--seed", SEED])
    best = [l for l in train_lines if "가장 좋은 검증 손실" in l]
    limit = ["--time-ms", 100000, "--depth", MATCH_DEPTH] if MATCH_DEPTH else ["--time-ms", MATCH_TIME_MS]
    match_lines = run(BIN[h] / "matchup", ["--a", W, "--net-a", net, "--b", W, "--games", MATCH_GAMES,
                                           *limit, "--threads", THREADS, "--seed", SEED + 1000])
    r = json.loads([l for l in match_lines if l.startswith("RESULT ")][-1][len("RESULT "):])
    results[h] = (best[-1].strip() if best else "", r)
    print(f"-> 은닉 {h}: {r['aWins']}승 {r['draws']}무 {r['bWins']}패, 점수 {r['score']:.3f} (구간 {r['low']:.3f} ~ {r['high']:.3f})")

## 7. 은닉 32

In [ ]:
if 32 in HIDDEN_LIST:
    run_one(32)
else:
    print('HIDDEN_LIST 에 32 가 없어서 건너뜀')

## 8. 은닉 256

In [ ]:
if 256 in HIDDEN_LIST:
    run_one(256)
else:
    print('HIDDEN_LIST 에 256 이 없어서 건너뜀')

## 9. 요약

In [ ]:
mode = f"깊이 {MATCH_DEPTH} 고정" if MATCH_DEPTH else f"수당 {MATCH_TIME_MS}ms"
print(f"=============== 요약 (신경망 vs 선형 평가, {mode}, 점수 0.5 = 동률) ===============")
for h, (loss, r) in results.items():
    print(f"은닉 {h:5d}: {r['aWins']}승 {r['draws']}무 {r['bWins']}패  점수 {r['score']:.3f} "
          f"(95% 구간 {r['low']:.3f} ~ {r['high']:.3f})")
    print(f"           {loss}")

## 10. 결과를 Google Drive 에 저장 (Claude 가 읽을 수 있게)
내 드라이브의 `slider_results` 폴더에 실험 설정이 이름에 들어간 txt 로 저장합니다. 로그인 창이 뜨면 허용하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")   # 로그인 창이 뜨면 허용

out_dir = pathlib.Path("/content/drive/MyDrive/slider_results"); out_dir.mkdir(parents=True, exist_ok=True)
mode = f"깊이{MATCH_DEPTH}" if MATCH_DEPTH else f"{MATCH_TIME_MS}ms"
name = f"slider_result_{GAMES}g_{TIME_MS}ms_match-{mode}.txt"

summary = {
    "settings": {"GAMES": GAMES, "TIME_MS": TIME_MS, "EPOCHS": EPOCHS, "MATCH_GAMES": MATCH_GAMES,
                 "MATCH_DEPTH": MATCH_DEPTH, "MATCH_TIME_MS": MATCH_TIME_MS, "SEED": SEED, "weights": W, "threads": THREADS},
    "selfplay_file": str(data), "selfplay_bytes": data.stat().st_size,
    "results": {str(h): {"train": results[h][0], "match": results[h][1]} for h in HIDDEN_LIST if h in results},
}
lines = [f"설정: {summary['settings']}", f"자가대전 데이터: {data.name} ({data.stat().st_size / 1e6:.1f}MB)"]
for h in HIDDEN_LIST:
    if h in results:
        loss, r = results[h]
        lines.append(f"은닉 {h}: {r['aWins']}승 {r['draws']}무 {r['bWins']}패  점수 {r['score']:.3f} (95% 구간 {r['low']:.3f} ~ {r['high']:.3f})")
        lines.append(f"  {loss}")
lines += ["", "JSON:", json.dumps(summary, ensure_ascii=False, indent=1)]
(out_dir / name).write_text("\n".join(lines), encoding="utf-8")
print("저장:", out_dir / name)
print("\n".join(lines[:6]))
